# Blob to Bronze — Transaction Ingestion

This notebook uses **Auto Loader** (`cloudFiles`) to read transaction CSV files from the source volume and writes them into the bronze Delta table `retail_q.blob_bronze.transactions`.

In [0]:
# Source and target configuration
source_path = "/Volumes/retail_q/vol_blob/blob_source/transaction_source/"
target_table = "retail_q.blob_bronze.transactions"
checkpoint_path = "/Volumes/retail_q/vol_blob/blob_source/_checkpoints/transactions"

# Read CSV files using Auto Loader
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
      .option("cloudFiles.inferColumnTypes", "true")
      .option("header", "true")
      .option("rescuedDataColumn", "_rescued_data")
      .load(source_path))

# Write to the bronze Delta table — Trigger.availableNow processes all existing files in one batch
query = (df.writeStream
         .format("delta")
         .option("checkpointLocation", f"{checkpoint_path}/checkpoint")
         .option("mergeSchema", "true")
         .trigger(availableNow=True)
         .toTable(target_table))

query.awaitTermination()
print("Auto Loader ingestion complete.")